# Library and my ip host

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, lit, schema_of_json
# from pyspark.sql.types import StructType, StructField, DateType, StringType, FloatType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, avg, min, max

In [3]:
spark_session = SparkSession.builder.appName("Average_calculation").remote("sc://spark-test:15002").getOrCreate()

In [4]:
import socket

hostname = socket.gethostname()
ip_address = socket.gethostbyname(hostname)

print(f"Tên máy: {hostname}")
print(f"Địa chỉ IP: {ip_address}")


Tên máy: DELL_VOSTRO14
Địa chỉ IP: 192.168.1.100


# ETL

In [ ]:
df = spark_session.read \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "kafka-test:9092") \
        .option("subscribe", "air_quality_data") \
        .option("startingOffsets", "earliest") \
        .option("endingOffsets", "latest") \
        .load()
df.show()

In [ ]:
object_air_data = df.withColumns(
    {
        "key": df["key"].cast("string"),
        "value": df["value"].cast("string")
    }
).select("value")

object_air_data.show()

+--------------------+
|               value|
+--------------------+
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
|{"district": "Ba ...|
+--------------------+
only showing top 20 rows


In [ ]:
import json

rows = object_air_data.select("value").limit(10).collect()
schema = schema_of_json(lit(json.dumps(json.loads(rows[9].asDict()["value"]))))
schema

Column<'schema_of_json({"district": "Ba Dinh", "city": "Ha Noi", "time": "2022-08-05T20:00", "pm10 (\u03bcg/m\u00b3)": 77.5, "pm2_5 (\u03bcg/m\u00b3)": 54.1, "carbon_monoxide (\u03bcg/m\u00b3)": 1027.0, "carbon_dioxide (ppm)": NaN, "nitrogen_dioxide (\u03bcg/m\u00b3)": 92.9, "sulphur_dioxide (\u03bcg/m\u00b3)": 28.6, "ozone (\u03bcg/m\u00b3)": 35.0, "aerosol_optical_depth ()": 0.67, "dust (\u03bcg/m\u00b3)": 0.0, "uv_index ()": 0.0, "uv_index_clear_sky ()": 0.0})'>

In [ ]:
df_parsed = object_air_data.withColumn("value", from_json(col("value"), schema))
df_parsed.show(5)

+--------------------+
|               value|
+--------------------+
|{NaN, NaN, NaN, H...|
|{NaN, NaN, NaN, H...|
|{NaN, NaN, NaN, H...|
|{0.94, NaN, 891.0...|
|{0.66, NaN, 328.0...|
+--------------------+
only showing top 5 rows


In [ ]:
analysis_df = df_parsed.select(
    "value.city",
    "value.district",
    "value.time",
    col("value.pm2_5 (μg/m³)").alias("pm25"),
    col("value.pm10 (μg/m³)").alias("pm10"),
    col("value.carbon_monoxide (μg/m³)").alias("co"),
    # col("value.carbon_dioxide (ppm)").alias("co2"),
    col("value.nitrogen_dioxide (μg/m³)").alias("no2"),
    col("value.sulphur_dioxide (μg/m³)").alias("so2"),
    col("value.ozone (μg/m³)").alias("o3"),
    col("value.aerosol_optical_depth ()").alias("aod"),
    col("value.dust (μg/m³)").alias("dust"),
    col("value.uv_index ()").alias("uv_idx"),
    col("value.uv_index_clear_sky ()").alias("uv_idx_clear_sky"),
).filter((col("pm25").isNaN() == False) & (col("pm10").isNaN() == False)).orderBy("value.time")

In [ ]:
analysis_df.write.format("csv").mode("overwrite").option("header", "true").save("hdfs://hadoop-test:9000/data/csv/processed/")

# Prepare data and save into sql database (datawarehouse) (postgresql)

In [3]:
test_df = spark_session.read.format("csv").option("header", "true").load("hdfs://hadoop-test:9000/data/csv/processed/")

In [4]:
def avarage_by(df, previous_rows_count, column_names):
    window = Window.orderBy("time").rowsBetween(1-previous_rows_count, 0)
    result = df
    for col_name in column_names:
        result = result.withColumn(f"avg_{col_name}_{previous_rows_count}h", avg(col_name).over(window))
    return result

In [5]:
test_df = avarage_by(test_df, 24, ["pm25", "pm10"])

In [6]:
test_df = avarage_by(test_df, 1, ["o3"])

In [7]:
test_df = avarage_by(test_df, 8, ["o3"])

In [8]:
test_df = avarage_by(test_df, 1, ["co", "so2", "no2"])

In [9]:
test_df.show(5)

d:\apps\conda\envs\mmd_env\Lib\site-packages\pyspark\sql\connect\expressions.py:1091: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+--------+----------------+----+----+-----+----+----+-----+----+----+------+----------------+------------------+------------------+---------+------------------+---------+----------+----------+
|  city|district|            time|pm25|pm10|   co| no2| so2|   o3| aod|dust|uv_idx|uv_idx_clear_sky|      avg_pm25_24h|      avg_pm10_24h|avg_o3_1h|         avg_o3_8h|avg_co_1h|avg_so2_1h|avg_no2_1h|
+------+--------+----------------+----+----+-----+----+----+-----+----+----+------+----------------+------------------+------------------+---------+------------------+---------+----------+----------+
|Ha Noi| Ba Dinh|2022-08-04T07:00|40.3|58.0|595.0|29.7|16.8| 24.0|0.43| 0.0|  0.75|             0.8|              40.3|              58.0|     24.0|              24.0|    595.0|      16.8|      29.7|
|Ha Noi| Ba Dinh|2022-08-04T08:00|30.0|43.5|552.0|25.0|18.2| 49.0|0.52| 0.0|   2.1|            2.35|             35.15|             50.75|     49.0|              36.5|    552.0|      18.2|      25.0|


In [ ]:
test_df.select("time", "avg_pm25_24h", "avg_pm10_24h", "avg_o3_1h", "avg_o3_8h", "avg_co_1h", "avg_so2_1h", "avg_no2_1h") \
        .write \
        .format("jdbc") \
        .option("url", f"jdbc:postgresql://{ip_address}:5432/air_quality_datawarehouse") \
        .option("dbtable", "average_values") \
        .option("user", "admin") \
        .option("password", "admin123") \
        .option("driver", "org.postgresql.Driver") \
        .option("batchsize", 1000) \
        .mode("overwrite") \
        .save()

# Analysis

## Metric table

In [5]:
import pandas as pd

bp_idx = [
    {"I": [0, 50, 100, 150, 200, 300, 400, 500]},
    {"BP_o3_1h": [0, 160, 200, 300, 400, 800, 1000, 1200]},
    {"BP_o3_8h": [0, 100, 120, 170, 210, 400]},
    {"BP_co": [0, 10000, 30000, 45000, 60000, 90000, 120000, 150000]},
    {"BP_so2": [0, 125, 350, 550, 800, 1600, 2100, 2630]},
    {"BP_no2": [0, 100, 200, 700, 1200, 2350, 3100, 3850]},
    {"BP_pm10": [0, 50, 150, 250, 350, 420, 500, 600]},
    {"BP_pm25": [0, 25, 50, 80, 150, 250, 350, 500]}
]
bp_dict = {list(d.keys())[0]: list(d.values())[0] for d in bp_idx}

df_bp = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in bp_dict.items()]))

df_bp = spark_session.createDataFrame(df_bp)

In [6]:
df_bp.write.format("jdbc") \
        .option("url", f"jdbc:postgresql://{ip_address}:5432/air_quality_datawarehouse") \
        .option("dbtable", "bp_index") \
        .option("user", "admin") \
        .option("password", "admin123") \
        .option("driver", "org.postgresql.Driver") \
        .option("batchsize", 1000) \
        .mode("overwrite") \
        .save()

## AQI calculation

### load

In [7]:
df_bp = spark_session.read.format("jdbc") \
        .option("url", f"jdbc:postgresql://{ip_address}:5432/air_quality_datawarehouse") \
        .option("dbtable", "bp_index") \
        .option("user", "admin") \
        .option("password", "admin123") \
        .option("driver", "org.postgresql.Driver") \
        .load()

df_bp = df_bp.orderBy(col("I"))

In [8]:
bp_o3_1h_df = df_bp.select("I", "BP_o3_1h")
bp_o3_8h_df = df_bp.select("I", "BP_o3_8h")
bp_co_df = df_bp.select("I", "BP_co")
bp_so2_df = df_bp.select("I", "BP_so2")
bp_no2_df = df_bp.select("I", "BP_no2")
bp_pm10_df = df_bp.select("I", "BP_pm10")
bp_pm25_df = df_bp.select("I", "BP_pm25")

In [9]:
average_df = spark_session.read.format("jdbc") \
        .option("url", f"jdbc:postgresql://{ip_address}:5432/air_quality_datawarehouse") \
        .option("dbtable", "average_values") \
        .option("user", "admin") \
        .option("password", "admin123") \
        .option("driver", "org.postgresql.Driver") \
        .load()

In [10]:
average_df.filter(col("time") == "2022-08-04T07:00").show(5)

+----------------+------------+------------+---------+---------+---------+----------+----------+
|            time|avg_pm25_24h|avg_pm10_24h|avg_o3_1h|avg_o3_8h|avg_co_1h|avg_so2_1h|avg_no2_1h|
+----------------+------------+------------+---------+---------+---------+----------+----------+
|2022-08-04T07:00|        40.3|        58.0|     24.0|     24.0|    595.0|      16.8|      29.7|
+----------------+------------+------------+---------+---------+---------+----------+----------+



In [11]:
avg_pm25_24h_df = average_df.select("time", "avg_pm25_24h")
avg_pm10_24h_df = average_df.select("time", "avg_pm10_24h")
avg_o3_1h_df = average_df.select("time", "avg_o3_1h")
avg_o3_8h_df = average_df.select("time", "avg_o3_8h")
avg_co_1h_df = average_df.select("time", "avg_co_1h")
avg_so2_1h_df = average_df.select("time", "avg_so2_1h")
avg_no2_1h_df = average_df.select("time", "avg_no2_1h")

### start calculating

In [12]:
min_dis_window = Window.partitionBy(col("time")).orderBy(col("dis"))

#### AQI calculation for PM2.5

In [13]:
join_avg_pm25_24h_df = avg_pm25_24h_df.alias("avg_pm25_24h_df").join(bp_pm25_df.alias("bp_pm25_df"), col("avg_pm25_24h_df.avg_pm25_24h") >= col("bp_pm25_df.BP_pm25"), how="left")
join_avg_pm25_24h_df = join_avg_pm25_24h_df.join(bp_pm25_df.alias("bp_pm25_df2"), col("avg_pm25_24h_df.avg_pm25_24h") < col("bp_pm25_df2.BP_pm25"), how="left")

In [14]:
join_avg_pm25_24h_df = join_avg_pm25_24h_df.withColumn("dis", col("bp_pm25_df2.BP_pm25") - col("bp_pm25_df.BP_pm25"))
join_avg_pm25_24h_df = join_avg_pm25_24h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_pm25_24h_df = join_avg_pm25_24h_df.drop(col("dis"), col("min_dis"))

In [15]:
join_avg_pm25_24h_df = join_avg_pm25_24h_df.withColumn("aqi_pm25_24h", ((col("avg_pm25_24h_df.avg_pm25_24h") - col("bp_pm25_df.BP_pm25")) * (col("bp_pm25_df2.I") - col("bp_pm25_df.I")) / (col("bp_pm25_df2.BP_pm25") - col("bp_pm25_df.BP_pm25")) + col("bp_pm25_df.I")))

In [16]:
join_avg_pm25_24h_df = join_avg_pm25_24h_df.withColumn("date", col("time").substr(1, 10))

In [17]:
max_aqi_window = Window.partitionBy(col("date")).orderBy(col("avg_pm25_24h_df.avg_pm25_24h").desc())

join_avg_pm25_24h_df = join_avg_pm25_24h_df.withColumn("max_avg_pm25_24h", max(col("avg_pm25_24h_df.avg_pm25_24h")).over(max_aqi_window)) \
                    .filter(col("max_avg_pm25_24h") == col("avg_pm25_24h_df.avg_pm25_24h")) \
                    .drop(col("max_avg_pm25_24h"))

In [18]:
join_avg_pm25_24h_df.show(2)

+----------------+-----------------+---+-------+---+-------+-----------------+----------+
|            time|     avg_pm25_24h|  I|BP_pm25|  I|BP_pm25|     aqi_pm25_24h|      date|
+----------------+-----------------+---+-------+---+-------+-----------------+----------+
|2022-08-04T23:00|46.64117647058825| 50|     25|100|     50| 93.2823529411765|2022-08-04|
|2022-08-05T00:00|46.58333333333334| 50|     25|100|     50|93.16666666666669|2022-08-05|
+----------------+-----------------+---+-------+---+-------+-----------------+----------+
only showing top 2 rows


#### AQI calculation for PM10

In [19]:
join_avg_pm10_24h_df = avg_pm10_24h_df.alias("avg_pm10_24h_df").join(bp_pm10_df.alias("bp_pm10_df"), col("avg_pm10_24h_df.avg_pm10_24h") >= col("bp_pm10_df.BP_pm10"), how="left")
join_avg_pm10_24h_df = join_avg_pm10_24h_df.join(bp_pm10_df.alias("bp_pm10_df2"), col("avg_pm10_24h_df.avg_pm10_24h") < col("bp_pm10_df2.BP_pm10"), how="left")

In [20]:
join_avg_pm10_24h_df = join_avg_pm10_24h_df.withColumn("dis", col("bp_pm10_df2.BP_pm10") - col("bp_pm10_df.BP_pm10"))
join_avg_pm10_24h_df = join_avg_pm10_24h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_pm10_24h_df = join_avg_pm10_24h_df.drop(col("dis"), col("min_dis"))

In [21]:
join_avg_pm10_24h_df = join_avg_pm10_24h_df.withColumn("aqi_pm10_24h", ((col("avg_pm10_24h_df.avg_pm10_24h") - col("bp_pm10_df.BP_pm10")) * (col("bp_pm10_df2.I") - col("bp_pm10_df.I")) / (col("bp_pm10_df2.BP_pm10") - col("bp_pm10_df.BP_pm10")) + col("bp_pm10_df.I")))

In [22]:
join_avg_pm10_24h_df = join_avg_pm10_24h_df.withColumn("date", col("time").substr(1, 10))

In [23]:
max_aqi_window = Window.partitionBy(col("date")).orderBy(col("avg_pm10_24h_df.avg_pm10_24h").desc())

join_avg_pm10_24h_df = join_avg_pm10_24h_df.withColumn("max_avg_pm10_24h", max(col("avg_pm10_24h_df.avg_pm10_24h")).over(max_aqi_window)) \
                    .filter(col("max_avg_pm10_24h") == col("avg_pm10_24h_df.avg_pm10_24h")) \
                    .drop(col("max_avg_pm10_24h"))

In [24]:
join_avg_pm10_24h_df.show(100)

+----------------+------------------+---+-------+---+-------+------------------+----------+
|            time|      avg_pm10_24h|  I|BP_pm10|  I|BP_pm10|      aqi_pm10_24h|      date|
+----------------+------------------+---+-------+---+-------+------------------+----------+
|2022-08-04T23:00| 67.08235294117648| 50|     50|100|    150| 58.54117647058824|2022-08-04|
|2022-08-05T00:00| 67.00555555555556| 50|     50|100|    150| 58.50277777777778|2022-08-05|
|2022-08-06T06:00| 54.96666666666666| 50|     50|100|    150|52.483333333333334|2022-08-06|
|2022-08-07T00:00| 39.09999999999999|  0|      0| 50|     50| 39.09999999999999|2022-08-07|
|2022-08-08T14:00| 45.11250000000001|  0|      0| 50|     50| 45.11250000000001|2022-08-08|
|2022-08-09T23:00| 50.15833333333333| 50|     50|100|    150|50.079166666666666|2022-08-09|
|2022-08-10T09:00| 67.96666666666667| 50|     50|100|    150|58.983333333333334|2022-08-10|
|2022-08-11T00:00| 47.36666666666667|  0|      0| 50|     50| 47.36666666666667|

#### AQI calculation for no2

In [25]:
join_avg_no2_1h_df = avg_no2_1h_df.alias("avg_no2_1h_df").join(bp_no2_df.alias("bp_no2_df"), col("avg_no2_1h_df.avg_no2_1h") >= col("bp_no2_df.BP_no2"), how="left")
join_avg_no2_1h_df = join_avg_no2_1h_df.join(bp_no2_df.alias("bp_no2_df2"), col("avg_no2_1h_df.avg_no2_1h") < col("bp_no2_df2.BP_no2"), how="left")

In [26]:
join_avg_no2_1h_df = join_avg_no2_1h_df.withColumn("dis", col("bp_no2_df2.BP_no2") - col("bp_no2_df.BP_no2"))
join_avg_no2_1h_df = join_avg_no2_1h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_no2_1h_df = join_avg_no2_1h_df.drop(col("dis"), col("min_dis"))

In [27]:
join_avg_no2_1h_df = join_avg_no2_1h_df.withColumn("aqi_no2", ((col("avg_no2_1h_df.avg_no2_1h") - col("bp_no2_df.BP_no2")) * (col("bp_no2_df2.I") - col("bp_no2_df.I")) / (col("bp_no2_df2.BP_no2") - col("bp_no2_df.BP_no2")) + col("bp_no2_df.I")))

In [28]:
join_avg_no2_1h_df = join_avg_no2_1h_df.withColumn("date", col("time").substr(1, 10))

In [29]:
max_aqi_window = Window.partitionBy(col("date")).orderBy(col("avg_no2_1h_df.avg_no2_1h").desc())

join_avg_no2_1h_df = join_avg_no2_1h_df.withColumn("max_avg_no2_1h", max(col("avg_no2_1h_df.avg_no2_1h")).over(max_aqi_window)) \
                    .filter(col("max_avg_no2_1h") == col("avg_no2_1h_df.avg_no2_1h")) \
                    .drop(col("max_avg_no2_1h"))

In [30]:
join_avg_no2_1h_df.show(2)

+----------------+----------+---+------+---+------+-------+----------+
|            time|avg_no2_1h|  I|BP_no2|  I|BP_no2|aqi_no2|      date|
+----------------+----------+---+------+---+------+-------+----------+
|2022-08-04T19:00|      78.9|  0|     0| 50|   100|  39.45|2022-08-04|
|2022-08-05T19:00|      97.8|  0|     0| 50|   100|   48.9|2022-08-05|
+----------------+----------+---+------+---+------+-------+----------+
only showing top 2 rows


#### AQI calculation for so2

In [31]:
join_avg_so2_1h_df = avg_so2_1h_df.alias("avg_so2_1h_df").join(bp_so2_df.alias("bp_so2_df"), col("avg_so2_1h_df.avg_so2_1h") >= col("bp_so2_df.BP_so2"), how="left")
join_avg_so2_1h_df = join_avg_so2_1h_df.join(bp_so2_df.alias("bp_so2_df2"), col("avg_so2_1h_df.avg_so2_1h") < col("bp_so2_df2.BP_so2"), how="left")

In [32]:
join_avg_so2_1h_df = join_avg_so2_1h_df.withColumn("dis", col("bp_so2_df2.BP_so2") - col("bp_so2_df.BP_so2"))
join_avg_so2_1h_df = join_avg_so2_1h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_so2_1h_df = join_avg_so2_1h_df.drop(col("dis"), col("min_dis"))

In [33]:
join_avg_so2_1h_df = join_avg_so2_1h_df.withColumn("aqi_so2", ((col("avg_so2_1h_df.avg_so2_1h") - col("bp_so2_df.BP_so2")) * (col("bp_so2_df2.I") - col("bp_so2_df.I")) / (col("bp_so2_df2.BP_so2") - col("bp_so2_df.BP_so2")) + col("bp_so2_df.I")))

In [34]:
join_avg_so2_1h_df = join_avg_so2_1h_df.withColumn("date", col("time").substr(1, 10))

In [35]:
max_aqi_window = Window.partitionBy(col("date")).orderBy(col("avg_so2_1h_df.avg_so2_1h").desc())

join_avg_so2_1h_df = join_avg_so2_1h_df.withColumn("max_avg_so2_1h", max(col("avg_so2_1h_df.avg_so2_1h")).over(max_aqi_window)) \
                    .filter(col("max_avg_so2_1h") == col("avg_so2_1h_df.avg_so2_1h")) \
                    .drop(col("max_avg_so2_1h"))

In [36]:
join_avg_so2_1h_df.show(2)

+----------------+----------+---+------+---+------+-------+----------+
|            time|avg_so2_1h|  I|BP_so2|  I|BP_so2|aqi_so2|      date|
+----------------+----------+---+------+---+------+-------+----------+
|2022-08-04T18:00|      22.9|  0|     0| 50|   125|   9.16|2022-08-04|
|2022-08-05T22:00|      32.0|  0|     0| 50|   125|   12.8|2022-08-05|
+----------------+----------+---+------+---+------+-------+----------+
only showing top 2 rows


#### AQI calculation for co

In [37]:
join_avg_co_1h_df = avg_co_1h_df.alias("avg_co_1h_df").join(bp_co_df.alias("bp_co_df"), col("avg_co_1h_df.avg_co_1h") >= col("bp_co_df.BP_co"), how="left")
join_avg_co_1h_df = join_avg_co_1h_df.join(bp_co_df.alias("bp_co_df2"), col("avg_co_1h_df.avg_co_1h") < col("bp_co_df2.BP_co"), how="left")

In [38]:
join_avg_co_1h_df = join_avg_co_1h_df.withColumn("dis", col("bp_co_df2.BP_co") - col("bp_co_df.BP_co"))
join_avg_co_1h_df = join_avg_co_1h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_co_1h_df = join_avg_co_1h_df.drop(col("dis"), col("min_dis"))

In [39]:
join_avg_co_1h_df = join_avg_co_1h_df.withColumn("aqi_co", (col("avg_co_1h_df.avg_co_1h") - col("bp_co_df.BP_co")) * (col("bp_co_df2.I") - col("bp_co_df.I")) / (col("bp_co_df2.BP_co") - col("bp_co_df.BP_co")) + col("bp_co_df.I"))

In [40]:
join_avg_co_1h_df = join_avg_co_1h_df.withColumn("date", col("time").substr(1, 10))

In [41]:
max_aqi_window = Window.partitionBy(col("date")).orderBy(col("avg_co_1h_df.avg_co_1h").desc())

join_avg_co_1h_df = join_avg_co_1h_df.withColumn("max_avg_co_1h", max(col("avg_co_1h_df.avg_co_1h")).over(max_aqi_window)) \
                    .filter(col("max_avg_co_1h") == col("avg_co_1h_df.avg_co_1h")) \
                    .drop(col("max_avg_co_1h"))

In [42]:
join_avg_co_1h_df.show(5)

+----------------+---------+---+-----+---+-----+------+----------+
|            time|avg_co_1h|  I|BP_co|  I|BP_co|aqi_co|      date|
+----------------+---------+---+-----+---+-----+------+----------+
|2022-08-04T19:00|    891.0|  0|    0| 50|10000| 4.455|2022-08-04|
|2022-08-05T19:00|   1055.0|  0|    0| 50|10000| 5.275|2022-08-05|
|2022-08-06T00:00|    866.0|  0|    0| 50|10000|  4.33|2022-08-06|
|2022-08-07T23:00|    629.0|  0|    0| 50|10000| 3.145|2022-08-07|
|2022-08-08T01:00|    732.0|  0|    0| 50|10000|  3.66|2022-08-08|
+----------------+---------+---+-----+---+-----+------+----------+
only showing top 5 rows


#### AQI calculation for o3_1h

In [43]:
join_avg_o3_1h_df = avg_o3_1h_df.alias("avg_o3_1h_df").join(bp_o3_1h_df.alias("bp_o3_1h_df"), col("avg_o3_1h_df.avg_o3_1h") >= col("bp_o3_1h_df.BP_o3_1h"), how="left")
join_avg_o3_1h_df = join_avg_o3_1h_df.join(bp_o3_1h_df.alias("bp_o3_1h_df2"), col("avg_o3_1h_df.avg_o3_1h") < col("bp_o3_1h_df2.BP_o3_1h"), how="left")

In [44]:
join_avg_o3_1h_df = join_avg_o3_1h_df.withColumn("dis", col("bp_o3_1h_df2.BP_o3_1h") - col("bp_o3_1h_df.BP_o3_1h"))
join_avg_o3_1h_df = join_avg_o3_1h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_o3_1h_df = join_avg_o3_1h_df.drop(col("dis"), col("min_dis"))

In [45]:
join_avg_o3_1h_df = join_avg_o3_1h_df.withColumn("aqi_o3_1h", (col("avg_o3_1h_df.avg_o3_1h") - col("bp_o3_1h_df.BP_o3_1h")) * (col("bp_o3_1h_df2.I") - col("bp_o3_1h_df.I")) / (col("bp_o3_1h_df2.BP_o3_1h") - col("bp_o3_1h_df.BP_o3_1h")) + col("bp_o3_1h_df.I"))

In [46]:
join_avg_o3_1h_df = join_avg_o3_1h_df.withColumn("date", col("time").substr(1, 10))

In [47]:
max_aqi_window = Window.partitionBy(col("date")).orderBy(col("avg_o3_1h_df.avg_o3_1h").desc())

join_avg_o3_1h_df = join_avg_o3_1h_df.withColumn("max_avg_o3_1h", max(col("avg_o3_1h_df.avg_o3_1h")).over(max_aqi_window)) \
                    .filter(col("max_avg_o3_1h") == col("avg_o3_1h_df.avg_o3_1h")) \
                    .drop(col("max_avg_o3_1h"))

In [48]:
join_avg_o3_1h_df.show(2)

+----------------+---------+---+--------+---+--------+---------+----------+
|            time|avg_o3_1h|  I|BP_o3_1h|  I|BP_o3_1h|aqi_o3_1h|      date|
+----------------+---------+---+--------+---+--------+---------+----------+
|2022-08-04T13:00|    194.0| 50|     160|100|     200|     92.5|2022-08-04|
|2022-08-05T15:00|    135.0|  0|       0| 50|     160|  42.1875|2022-08-05|
+----------------+---------+---+--------+---+--------+---------+----------+
only showing top 2 rows


#### AQI calculation for o3_8h

In [49]:
join_avg_o3_8h_df = avg_o3_8h_df.alias("avg_o3_8h_df").join(bp_o3_8h_df.alias("bp_o3_8h_df"), col("avg_o3_8h_df.avg_o3_8h") >= col("bp_o3_8h_df.BP_o3_8h"), how="left")
join_avg_o3_8h_df = join_avg_o3_8h_df.join(bp_o3_8h_df.alias("bp_o3_8h_df2"), col("avg_o3_8h_df.avg_o3_8h") < col("bp_o3_8h_df2.BP_o3_8h"), how="left")

In [50]:
join_avg_o3_8h_df = join_avg_o3_8h_df.withColumn("dis", col("bp_o3_8h_df2.BP_o3_8h") - col("bp_o3_8h_df.BP_o3_8h"))
join_avg_o3_8h_df = join_avg_o3_8h_df.withColumn("min_dis", min(col("dis")).over(min_dis_window)).filter(col("min_dis") == col("dis"))
join_avg_o3_8h_df = join_avg_o3_8h_df.drop(col("dis"), col("min_dis"))

In [51]:
join_avg_o3_8h_df = join_avg_o3_8h_df.withColumn("aqi_o3_8h", (col("avg_o3_8h_df.avg_o3_8h") - col("bp_o3_8h_df.BP_o3_8h")) * (col("bp_o3_8h_df2.I") - col("bp_o3_8h_df.I")) / (col("bp_o3_8h_df2.BP_o3_8h") - col("bp_o3_8h_df.BP_o3_8h")) + col("bp_o3_8h_df.I"))

In [52]:
join_avg_o3_8h_df = join_avg_o3_8h_df.withColumn("date", col("time").substr(1, 10))

In [53]:
max_aqi_window = Window.partitionBy(col("date")).orderBy(col("avg_o3_8h_df.avg_o3_8h").desc())

join_avg_o3_8h_df = join_avg_o3_8h_df.withColumn("max_avg_o3_8h", max(col("avg_o3_8h_df.avg_o3_8h")).over(max_aqi_window)) \
                    .filter(col("max_avg_o3_8h") == col("avg_o3_8h_df.avg_o3_8h")) \
                    .drop(col("max_avg_o3_8h"))

In [54]:
join_avg_o3_8h_df.show(2)

+----------------+---------+---+--------+---+--------+---------+----------+
|            time|avg_o3_8h|  I|BP_o3_8h|  I|BP_o3_8h|aqi_o3_8h|      date|
+----------------+---------+---+--------+---+--------+---------+----------+
|2022-08-04T17:00|  160.125|100|   120.0|150|   170.0|  140.125|2022-08-04|
|2022-08-05T17:00|    114.5| 50|   100.0|100|   120.0|    86.25|2022-08-05|
+----------------+---------+---+--------+---+--------+---------+----------+
only showing top 2 rows
